# 03b Advanced Sentiment & Topic Modeling

回应 proposal RQ2(对比多种 sentiment 方法)与 "topic modeling" 计划。

本 notebook 在 03 的 dictionary baseline 之上额外加:
- **VADER** — 通用 sentiment 工具
- **FinBERT** — 金融领域专用预训练模型(ProsusAI/finbert)
- **LDA Topic Modeling** — 把 headline 聚成 8 个主题,作为辅助特征

输出:
- `data/processed/headlines_with_sentiment_v2.csv`(headline 级,含三种 sentiment)
- `data/processed/daily_with_sentiment_v2.csv`(日级,所有方法 + 主题分布合并)


## 1. 加载与设置

In [1]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import time
import json
import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
DATA = ROOT / 'data' / 'processed'
print('数据目录:', DATA)

# 读 03 已经清洗过的 headline 表
heads = pd.read_csv(DATA / 'headlines_with_sentiment.csv')
print('headlines 行数:', len(heads))
heads.head(3)


数据目录: C:\Users\15759\Desktop\sp500-main\data\processed
headlines 行数: 18153


,Title,Date,CP,token_count,general_positive_count,general_negative_count,finance_positive_count,finance_negative_count,general_sentiment_score,finance_sentiment_score,finance_sentiment_label
0,2008 predictions for the S&P 500,2008-01-02,1447.16,7,0,0,0,0,0.0,0.0,neutral
1,Dow Tallies Biggest First-session-of-year Poin...,2008-01-02,1447.16,10,0,1,0,1,-0.1,-0.1,negative
2,"JPMorgan Predicts 2008 Will Be ""Nothing But Net""",2008-01-02,1447.16,8,0,0,0,0,0.0,0.0,neutral


## 2. VADER — 通用 sentiment(回答 RQ2)

VADER 是基于规则 + 词典的通用情感分析工具,常用作 NLP baseline。

In [2]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()
def vader_scores(text):
    s = vader.polarity_scores(str(text))
    return s['compound'], s['pos'], s['neg'], s['neu']

t0 = time.time()
vs = heads['Title'].astype(str).apply(vader_scores)
heads['vader_compound'] = [x[0] for x in vs]
heads['vader_pos']      = [x[1] for x in vs]
heads['vader_neg']      = [x[2] for x in vs]
heads['vader_neu']      = [x[3] for x in vs]
print(f'VADER 耗时 {time.time()-t0:.1f}s')
heads[['Title','vader_compound']].head()


VADER 耗时 0.5s


,Title,vader_compound
0,2008 predictions for the S&P 500,0.0000
1,Dow Tallies Biggest First-session-of-year Poin...,-0.2732
2,"JPMorgan Predicts 2008 Will Be ""Nothing But Net""",0.0000
3,"U.S. Stocks Higher After Economic Data, Monsan...",0.0000
4,U.S. Stocks Climb As Hopes Increase For More F...,0.6249


## 3. FinBERT — 金融领域 sentiment

FinBERT (ProsusAI/finbert) 是在金融语料上 fine-tune 的 BERT 变体,直接输出 positive / negative / neutral 概率。

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tok = AutoTokenizer.from_pretrained('ProsusAI/finbert')
mdl = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert')
mdl.eval()
id2label = mdl.config.id2label
print('FinBERT labels:', id2label)


FinBERT labels: {0: 'positive', 1: 'negative', 2: 'neutral'}


In [4]:
# 批量推理函数 (CPU,batch_size=32)
def finbert_score_batch(texts, batch_size=32):
    probs_all = []
    n = len(texts)
    for i in range(0, n, batch_size):
        batch = texts[i:i+batch_size]
        enc = tok(batch, padding=True, truncation=True, max_length=64, return_tensors='pt')
        with torch.no_grad():
            logits = mdl(**enc).logits
            probs = torch.softmax(logits, dim=-1).numpy()
        probs_all.append(probs)
        if (i // batch_size) % 50 == 0:
            print(f'  进度 {i}/{n}', flush=True)
    return np.vstack(probs_all)

t0 = time.time()
texts = heads['Title'].astype(str).tolist()
probs = finbert_score_batch(texts, batch_size=32)
print(f'FinBERT 总耗时 {(time.time()-t0)/60:.1f} 分钟')

# 写回 — id2label = {0:'positive',1:'negative',2:'neutral'}
label_to_idx = {v:k for k,v in id2label.items()}
heads['finbert_p_pos'] = probs[:, label_to_idx['positive']]
heads['finbert_p_neg'] = probs[:, label_to_idx['negative']]
heads['finbert_p_neu'] = probs[:, label_to_idx['neutral']]
heads['finbert_score'] = heads['finbert_p_pos'] - heads['finbert_p_neg']
heads['finbert_label'] = probs.argmax(axis=1)
heads['finbert_label'] = heads['finbert_label'].map(id2label)
heads[['Title','finbert_label','finbert_score']].head()


  进度 0/18153


  进度 1600/18153


  进度 3200/18153


  进度 4800/18153


  进度 6400/18153


...

  进度 14400/18153


  进度 16000/18153


  进度 17600/18153


FinBERT 总耗时 4.6 分钟


,Title,finbert_label,finbert_score
0,2008 predictions for the S&P 500,neutral,-0.069876
1,Dow Tallies Biggest First-session-of-year Poin...,negative,-0.950723
2,"JPMorgan Predicts 2008 Will Be ""Nothing But Net""",negative,-0.415776
3,"U.S. Stocks Higher After Economic Data, Monsan...",positive,0.797434
4,U.S. Stocks Climb As Hopes Increase For More F...,positive,0.421511


## 4. LDA 主题建模

把 18k 条 headline 聚成 8 个主题,看哪些主题与 next-day 收益更相关。

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# 用 1-2 gram 构建词项矩阵
vec = CountVectorizer(max_df=0.7, min_df=20, stop_words='english',
                      ngram_range=(1,2), max_features=5000)
X = vec.fit_transform(heads['Title'].astype(str))
print('TDM shape:', X.shape)

K = 8
lda = LatentDirichletAllocation(n_components=K, learning_method='batch',
                                random_state=42, max_iter=20, n_jobs=-1)
t0 = time.time()
topic_dist = lda.fit_transform(X)
print(f'LDA 耗时 {time.time()-t0:.1f}s, perplexity={lda.perplexity(X):.0f}')

# 保存每个主题的关键词,用作主题命名参考
words = vec.get_feature_names_out()
print('\nTop words per topic:')
for k in range(K):
    top = lda.components_[k].argsort()[-8:][::-1]
    print(f'  Topic {k}: {", ".join(words[i] for i in top)}')

for k in range(K):
    heads[f'topic_{k}'] = topic_dist[:, k]


TDM shape: (18153, 1313)


LDA 耗时 11.8s, perplexity=444

Top words per topic:
  Topic 0: 500, nasdaq, dow, stocks, jones, dow jones, fed, 100
  Topic 1: 500, market, says, bear, index, new, bull, history
  Topic 2: 500, stocks, year, 10, best, years, worst, buy
  Topic 3: dow, today, live, 500, updates, live updates, day, gains
  Topic 4: 500, etfs, index, etf, stocks, tech, companies, fund
  Topic 5: markets, stocks, buffett, warren, warren buffett, world, funds, recession
  Topic 6: 500, wall, street, wall street, high, record, report, stocks
  Topic 7: stock, market, stock market, news, market news, 2022, 2021, 2023


## 5. 保存 headline 级丰富表

In [6]:
out_h = DATA / 'headlines_with_sentiment_v2.csv'
heads.to_csv(out_h, index=False)
print(f'写入 {out_h}  {len(heads)} 行  {heads.shape[1]} 列')


写入 C:\Users\15759\Desktop\sp500-main\data\processed\headlines_with_sentiment_v2.csv  18153 行  28 列


## 6. 聚合到日级,合并到 daily_with_sentiment.csv

In [7]:
# 6.1 按日聚合 VADER / FinBERT / Topic
daily_v2 = heads.groupby('Date').agg(
    vader_compound_mean=('vader_compound', 'mean'),
    vader_compound_std =('vader_compound', 'std'),
    vader_pos_share    =('vader_compound', lambda s: (s > 0.05).mean()),
    vader_neg_share    =('vader_compound', lambda s: (s < -0.05).mean()),
    finbert_score_mean =('finbert_score',  'mean'),
    finbert_score_std  =('finbert_score',  'std'),
    finbert_pos_share  =('finbert_label',  lambda s: (s == 'positive').mean()),
    finbert_neg_share  =('finbert_label',  lambda s: (s == 'negative').mean()),
).reset_index()

# 6.2 主题日均值
topic_cols = [f'topic_{k}' for k in range(8)]
topic_daily = heads.groupby('Date')[topic_cols].mean().reset_index()
daily_v2 = daily_v2.merge(topic_daily, on='Date', how='left')

# 6.3 合并到既有 daily_with_sentiment.csv
old = pd.read_csv(DATA / 'daily_with_sentiment.csv', parse_dates=['Date'])
daily_v2['Date'] = pd.to_datetime(daily_v2['Date'])
merged = old.merge(daily_v2, on='Date', how='left')
print('合并后行数 / 列数:', merged.shape)
print('新增列:', [c for c in merged.columns if c not in old.columns][:10], '...')

out = DATA / 'daily_with_sentiment_v2.csv'
merged.to_csv(out, index=False)
print(f'写入 {out}')


合并后行数 / 列数: (3506, 48)
新增列: ['vader_compound_mean', 'vader_compound_std', 'vader_pos_share', 'vader_neg_share', 'finbert_score_mean', 'finbert_score_std', 'finbert_pos_share', 'finbert_neg_share', 'topic_0', 'topic_1'] ...
写入 C:\Users\15759\Desktop\sp500-main\data\processed\daily_with_sentiment_v2.csv


## 7. 描述性对比 — 三种 sentiment 与 next-day return 的相关性

In [8]:
# 7.1 同期相关
cmp_cols = ['avg_finance_sentiment','avg_general_sentiment',
            'vader_compound_mean','finbert_score_mean']
target = 'return_next_day'
corr = merged[cmp_cols + [target]].corr()[target].drop(target)
print('=== 各 sentiment 方法与 next-day return 的同期相关系数 ===')
print(corr.round(4).to_string())


=== 各 sentiment 方法与 next-day return 的同期相关系数 ===
avg_finance_sentiment   -0.0120
avg_general_sentiment   -0.0077
vader_compound_mean      0.0239
finbert_score_mean       0.0186
